# Notebook 01 - Financial ML Data Preparation

هدف:
- خواندن فایل‌های خام سهام از raw_data
- اعتبارسنجی ستون‌های مورد نیاز
- جداسازی داده‌های مدل‌سازی از داده‌های ارزیابی
- حفظ dEven برای split زمانی در مراحل بعد

نکته:
- dEven و insCode فقط برای کنترل داده و validation استفاده می‌شوند.
- ستون‌های ارزیابی هرگز وارد مدل نمی‌شوند.
- RSI_Signal از لیست ویژگی‌ها حذف شده است.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
ROOT = Path(r"E:\Iran_stock_trade")

RAW_DATA = ROOT / "raw_data"

OUTPUT_MODEL = ROOT / "data_ready" / "prepared"
OUTPUT_EVAL = ROOT / "data_ready" / "evaluation"

OUTPUT_MODEL.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL.mkdir(parents=True, exist_ok=True)

print("Raw path:", RAW_DATA)
print("Model output:", OUTPUT_MODEL)
print("Evaluation output:", OUTPUT_EVAL)

Raw path: E:\Iran_stock_trade\raw_data
Model output: E:\Iran_stock_trade\data_ready\prepared
Evaluation output: E:\Iran_stock_trade\data_ready\evaluation


In [ ]:
META_COLUMNS = [
    "insCode",
    "dEven"
]


FEATURE_COLUMNS = [
    "adj_open",
    "adj_high",
    "adj_low",
    "adj_last_price",

    "zigzag_up_new_2",
    "zigzag_down_new_2",

    "EMA_3",
    "EMA_5",
    "EMA_8",
    "EMA_10",
    "EMA_12",
    "EMA_15",

    "EMA_30",
    "EMA_35",
    "EMA_40",
    "EMA_45",
    "EMA_50",
    "EMA_60",
    "EMA_200",

    "RSI_14",

    "macd",

    "power_of_buy",
    "volume_per_avg_month",

    "ho_buy",
    "ho_sell",

    "x",
    "y",
    "color",

    "gmma",
    "started",
    "body"
]


TARGET_COLUMN = [
    "class"
]


EVALUATION_COLUMNS = [
    "max_price_find",
    "min_price_find",
    "after_days",
    "days_max_find"
]


ALL_REQUIRED = META_COLUMNS + FEATURE_COLUMNS + TARGET_COLUMN + EVALUATION_COLUMNS

print("Features:", len(FEATURE_COLUMNS))

Features: 31


In [ ]:
csv_files = list(RAW_DATA.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

Number of CSV files: 825


In [ ]:
sample_file = csv_files[0]

df_sample = pd.read_csv(sample_file)

print(sample_file.name)
print(df_sample.shape)

df_sample.head()

آ س پ.csv
(2683, 85)


,Unnamed: 0,priceChange,priceMin,priceMax,priceYesterday,priceFirst,last,id,insCode,dEven,...,y,color,gmma,started,body,class,max_price_find,min_price_find,after_days,days_max_find
0,0,370.0,12820.0,12820.0,12450.0,12820.0,False,0,17617474823279712,20260701,...,0.734872,2,1,6,1.000000,0,0.0,0.0,0.0,0
1,1,360.0,12450.0,12450.0,12090.0,12450.0,False,0,17617474823279712,20260630,...,0.739730,2,1,5,1.000000,0,0.0,0.0,0.0,0
2,2,350.0,12090.0,12090.0,11740.0,12090.0,False,0,17617474823279712,20260629,...,0.468127,2,1,4,1.000000,0,0.0,0.0,0.0,0
3,3,340.0,11230.0,11800.0,11460.0,11720.0,False,0,17617474823279712,20260628,...,0.867561,2,1,3,0.140351,0,0.0,0.0,0.0,0
4,4,330.0,11290.0,11460.0,11130.0,11290.0,False,0,17617474823279712,20260627,...,0.818134,2,1,2,1.000000,0,0.0,0.0,0.0,0


In [ ]:
def prepare_stock_file(file_path):

    df = pd.read_csv(file_path)

    missing_columns = [
        c for c in ALL_REQUIRED
        if c not in df.columns
    ]

    if missing_columns:
        return None, None, missing_columns


    model_df = df[
        META_COLUMNS +
        FEATURE_COLUMNS +
        TARGET_COLUMN
    ].copy()


    eval_df = df[
        META_COLUMNS +
        EVALUATION_COLUMNS
    ].copy()


    return model_df, eval_df, None

In [ ]:
success = 0
errors = []

for file in csv_files:

    model_df, eval_df, error = prepare_stock_file(file)

    if error:
        errors.append({
            "file": file.name,
            "missing_columns": error
        })
        continue


    name = file.stem

    model_df.to_csv(
        OUTPUT_MODEL / f"{name}_model.csv",
        index=False,
        encoding="utf-8-sig"
    )


    eval_df.to_csv(
        OUTPUT_EVAL / f"{name}_eval.csv",
        index=False,
        encoding="utf-8-sig"
    )

    success += 1


print("Completed:", success)
print("Errors:", len(errors))

Completed: 825
Errors: 0


In [ ]:
pd.DataFrame(errors)

""


In [ ]:
prepared_files = list(OUTPUT_MODEL.glob("*_model.csv"))
eval_files = list(OUTPUT_EVAL.glob("*_eval.csv"))

print("Prepared:", len(prepared_files))
print("Evaluation:", len(eval_files))

Prepared: 825
Evaluation: 825
